# Figures for §7.1 — the synthetic evidence

Produces two figures for the submission document:

| output | placeholder it fills |
|---|---|
| `synthetic-kpi-bars.pdf` / `.png` | *KPI bar chart, policy × KPI* |
| `synthetic-window-sweep.pdf` / `.png` | *window sweep: L1−L0 ride vs window width* |
| `synthetic-kpi-table.csv` / `.md` | the same numbers as a table, for the text |

**No CARLA server, no re-simulation.** Everything is read from
`results/runs`, which already holds the campaign. Running this notebook end
to end takes seconds and is safe while a CARLA campaign is writing to the
same store — it only reads.

Every figure is written as **PDF and PNG**. Use the **PDF** in the document:
it is vector, so it stays sharp at any size, and its labels remain real text.
Nothing here is interactive.

## The one filter that matters

The campaign is 2,280 runs on the 21-stop campus map (seed 42), 60 seeds,
19 policies, two patience regimes. The figures below use the **impatient**
half (patience 900 s) — with unlimited patience every policy serves 100% and
coverage discriminates nothing.

The store also holds runs from the load-crossover notebooks at *the same*
map, capacity, volume and arrival rate, differing only in a `max_time`
ceiling that neither experiment ever reaches. They must be excluded by hand,
and `campaign_runs()` does; the assertion under it is what makes a silent
re-pooling loud.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cycler import cycler
from scipy import stats

from src.experiment.store import DEFAULT_RUNS_DIR, load_frame

from IPython.display import Markdown, display

# Where the submission's figures are written. Outside the repo, beside the
# document that embeds them, so a figure and the prose quoting it move
# together. Override with AUTOBUS_FIGURES to render somewhere disposable.
FIGURES = Path(os.environ.get(
    "AUTOBUS_FIGURES",
    Path.home() / "Desktop" / "האוניברסיטה הפתוחה" / "סדנה"
    / "submission prep docs" / "figures"))
FIGURES.mkdir(parents=True, exist_ok=True)

# The campaign's palette, so every figure in the submission reads as one set.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#8b5cf6", "#d4a017"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT, "xtick.labelsize": 9,
    "ytick.labelsize": 9, "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 110,
    # Text stays text in the PDF, so the document can be searched and the
    # figure scales without going fuzzy.
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save(figure, name: str) -> None:
    """
    Write one figure as both a vector PDF and a raster PNG.

    The PDF is what belongs in the submission document -- it stays sharp at
    any zoom and its labels remain selectable text. The PNG is for anything
    that cannot embed a PDF, such as slides or a video frame.

    :param figure: The matplotlib figure to write.
    :param name: Base filename, without an extension.
    :return: None.
    """
    figure.tight_layout()
    for suffix in ("pdf", "png"):
        path = FIGURES / f"{name}.{suffix}"
        figure.savefig(path, format=suffix, dpi=300, bbox_inches="tight")
    print(f"wrote {FIGURES / name}.pdf and .png")


def as_markdown(frame: pd.DataFrame) -> str:
    """
    Render a table as a Markdown block.

    Written out by hand rather than with ``DataFrame.to_markdown``, which
    needs the ``tabulate`` package -- not a dependency worth adding to the
    environment to format six rows.

    :param frame: The table to render.
    :return: A Markdown table, index included as the first column.
    """
    index_name = frame.index.name or ""
    header = [str(index_name)] + [str(c) for c in frame.columns]
    lines = ["| " + " | ".join(header) + " |",
             "|" + "|".join("---" for _ in header) + "|"]
    for key, row in frame.iterrows():
        cells = [f"{value:,.2f}" if isinstance(value, (int, float))
                 else str(value) for value in row]
        lines.append("| " + " | ".join([str(key)] + cells) + " |")
    return "\n".join(lines) + "\n"


def save_table(frame: pd.DataFrame, name: str) -> None:
    """
    Write one table as CSV and as a Markdown block ready to paste.

    :param frame: The table to write.
    :param name: Base filename, without an extension.
    :return: None.
    """
    frame.to_csv(FIGURES / f"{name}.csv")
    (FIGURES / f"{name}.md").write_text(as_markdown(frame), encoding="utf-8")
    print(f"wrote {FIGURES / name}.csv and .md")


def ci_half_width(values: pd.Series, confidence: float = 0.95) -> float:
    """
    Half-width of the confidence interval on a mean.

    The design review's SS3.2 bar is stated on this quantity, so it is
    computed the same way here as in ``summarize_comparison``.

    :param values: The per-seed observations.
    :param confidence: Confidence level.
    :return: Half-width in the units of ``values``, or nan if n < 2.
    """
    n = values.count()
    if n < 2:
        return float("nan")
    return float(stats.t.ppf(0.5 + confidence / 2, n - 1)
                 * values.std(ddof=1) / np.sqrt(n))


print("figures will be written to:", FIGURES)

## The campaign

In [ ]:
def campaign_runs() -> pd.DataFrame:
    """
    The 2,280-run single-map campaign, and nothing else.

    Every term here is load-bearing. ``max_time`` in particular: the
    load-crossover notebooks ran the same map, capacity, volume and rate at
    a 200,000 s ceiling instead of 60,000 s, and because a ceiling neither
    run reaches changes no behaviour, dropping the term silently pools two
    experiments and double-counts seeds 0-29 on three of the arms.

    :return: One row per run.
    """
    frame = load_frame(DEFAULT_RUNS_DIR)
    return frame[
        (frame["identity.world.map"] == "planar")
        & (frame["identity.world.stop_density"] == 0.5)
        & (frame["identity.world.map_seed"] == 42)
        & (frame["identity.trial.capacity"] == 40)
        & (frame["identity.trial.max_requests"] == 200)
        & (frame["identity.trial.demand_interval_s"] == 60.0)
        & (frame["identity.trial.max_time"] == 60000.0)
        & (frame["identity.trial.max_wait_s"] == 900.0)
        & (frame["identity.seed"] < 60)
    ]


runs = campaign_runs()
print(f"{len(runs)} runs, {runs['identity.seed'].nunique()} seeds, "
      f"{runs['identity.policy'].nunique()} policies")
assert len(runs) == 1140, (
    f"expected the 1,140-run impatient half of the campaign, got {len(runs)}")

## Figure 1 — KPIs by policy

Four panels, one per design-review KPI, with **95% confidence intervals** on
every bar. Six arms rather than all nineteen: the two non-responsive
baselines, the two greedy policies that look good until coverage is read
beside them, and the two insertion families at the recommended window.

The panel arrangement is the argument. Read **served fraction first** — then
notice that `Nearest unserved` posts the *best* wait time in the field while
serving 20 points less than the winner. That is the KPI trap from §6.3, made
visible rather than asserted: a policy that abandons its hard passengers is
rewarded on the mean of those it kept.

In [ ]:
# Six arms, not nineteen. Ordered worst-to-best on coverage so the bars
# read bottom-up as a ladder, which is also the order the section argues in.
ARMS = ["Fixed route", "Skip-stop", "Nearest unserved", "Deviated",
        "Window W900", "Reordering W900"]

GATE = 80.0

# (column, label, unit, higher_is_better, format)
KPIS = [
    ("kpi.served_fraction", "Served fraction", "%", True, "{:.1f}"),
    ("kpi.mean_wait_time", "Mean wait time", "s", False, "{:.0f}"),
    ("kpi.mean_ride_time", "Mean ride time", "s", False, "{:.0f}"),
    ("kpi.throughput", "Throughput", "req/min", True, "{:.3f}"),
]

WINNER, TRAP, MUTED = SERIES[2], SERIES[4], "#b9b8b4"


def kpi_summary(campaign: pd.DataFrame) -> pd.DataFrame:
    """
    Mean and CI half-width for every arm and KPI.

    :param campaign: The campaign's runs.
    :return: Indexed by policy, one mean and one half-width per KPI.
    """
    rows = {}
    for arm in ARMS:
        block = campaign[campaign["identity.policy"] == arm]
        row = {}
        for column, label, _, _, _ in KPIS:
            values = block[column]
            scale = 100.0 if column == "kpi.served_fraction" else 1.0
            row[f"{label} (mean)"] = values.mean() * scale
            row[f"{label} (95% CI +/-)"] = ci_half_width(values) * scale
        rows[arm] = row
    return pd.DataFrame(rows).T.loc[ARMS]


summary = kpi_summary(runs)
coverage = summary["Served fraction (mean)"]

figure, axes = plt.subplots(2, 2, figsize=(10.8, 6.6))
positions = np.arange(len(ARMS))
for axis, (column, label, unit, higher_better, fmt) in zip(axes.ravel(),
                                                           KPIS):
    means = summary[f"{label} (mean)"].to_numpy()
    errors = summary[f"{label} (95% CI +/-)"].to_numpy()
    best = int(np.argmax(means) if higher_better else np.argmin(means))
    # Amber, not green, when the arm that wins a panel is one that does not
    # clear the fulfilment gate. Painting it "best" without qualification is
    # the exact misreading section 6.3 warns about, so the chart marks it.
    honest = coverage.iloc[best] >= GATE
    colours = [MUTED] * len(ARMS)
    colours[best] = WINNER if honest else TRAP
    axis.barh(positions, means, xerr=errors, color=colours, height=0.66,
              error_kw=dict(ecolor=INK_SOFT, capsize=3, lw=1.0))
    axis.set_yticks(positions)
    axis.set_yticklabels(ARMS)
    axis.invert_yaxis()
    axis.set_title(f"{label} ({unit})")
    axis.set_axisbelow(True)
    axis.grid(axis="x")
    span = float(means.max()) or 1.0
    axis.set_xlim(0, span * 1.30)
    for y, value in zip(positions, means):
        axis.text(value + span * 0.035, y, fmt.format(value), va="center",
                  fontsize=8.5, color=INK_SOFT)
    if not honest:
        # Top-right: the only corner guaranteed free, since the longest
        # bar in a panel whose winner is the shortest one is at the top.
        axis.text(0.985, 0.97,
                  "amber: best here, but serves only "
                  f"{coverage.iloc[best]:.1f}%",
                  transform=axis.transAxes, ha="right", va="top",
                  fontsize=8, color=TRAP)

gate_axis = axes[0][0]
gate_axis.axvline(GATE, ls="--", lw=1.1, color=INK_SOFT)
gate_axis.text(GATE + 1.5, -0.42, "80% gate", fontsize=8, color=INK_SOFT,
               va="center")
figure.suptitle("Campus map (21 stops), 60 seeds, patience 900 s — "
                "bars are 95% confidence intervals",
                fontsize=10, color=INK_SOFT, y=1.01)
save(figure, "synthetic-kpi-bars")
plt.show()

display(Markdown("**The same numbers, for the text.**"))
display(summary.round(3))
save_table(summary.round(3), "synthetic-kpi-table")

In [ ]:
# The claims §7.1 and §6.3 make about this figure, checked rather than
# trusted -- if the store changes under them, this cell is what says so.
served = summary["Served fraction (mean)"]
wait = summary["Mean wait time (mean)"]
worst_ci = (summary["Served fraction (95% CI +/-)"]
            / summary["Served fraction (mean)"] * 100).max()

checks = {
    "Reordering W900 wins on coverage":
        served.idxmax() == "Reordering W900",
    "Fixed route misses the 80% gate": served["Fixed route"] < 80,
    "Reordering W900 clears it": served["Reordering W900"] >= 80,
    "a greedy policy posts the best wait":
        wait.idxmin() == "Nearest unserved",
    "...while serving less than the winner":
        served["Nearest unserved"] < served["Reordering W900"],
    "every arm meets the 5% CI bar on coverage": worst_ci < 5.0,
}
for claim, holds in checks.items():
    print(f"  {'OK  ' if holds else 'FAIL'}  {claim}")
print(f"\nworst coverage CI half-width: {worst_ci:.2f}% of its mean")
print(f"wait gap, best-waiting policy vs the coverage winner: "
      f"{wait['Nearest unserved']:.0f} s at "
      f"{served['Nearest unserved']:.1f}% served, against "
      f"{wait['Reordering W900']:.0f} s at "
      f"{served['Reordering W900']:.1f}%")
assert all(checks.values()), "a claim in SS7.1 no longer holds -- read above"

## Figure 2 — what reordering buys, against the service window

L1 (`ReorderingInsertionPolicy`) is L0 plus one repair pass over the order
already planned. The question this figure answers is *when that pass is worth
running*, and the answer is not flat: **the advantage grows as the window
tightens.**

Paired per seed — the same 60 demand draws feed both arms at each window, so
the difference is the policy and not the sample. Error bars are 95% CIs on
the paired difference; a bar clear of the zero line is a real effect.

The mechanism worth being able to say out loud: a tighter deadline constrains
the schedule harder, and repairing the order pays exactly when there is less
freedom to absorb a bad insertion. The original single-window ablation could
not have seen this, because it only ever looked at one point on this axis.

In [ ]:
WINDOWS = [450, 600, 750, 900, 1200, 1800]


def paired_gap(runs: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    L1 minus L0 at each window, paired by seed.

    :param runs: The campaign's runs.
    :param column: KPI column to difference.
    :return: One row per window: mean difference, CI half-width, p-value.
    """
    rows = []
    for window in WINDOWS:
        wide = runs[runs["identity.policy"].isin(
            [f"Reordering W{window}", f"Window W{window}"])].pivot_table(
                index="identity.seed", columns="identity.policy",
                values=column).dropna()
        difference = (wide[f"Reordering W{window}"]
                      - wide[f"Window W{window}"])
        _, p_value = stats.ttest_rel(wide[f"Reordering W{window}"],
                                     wide[f"Window W{window}"])
        rows.append({"window_s": window, "mean_gap": difference.mean(),
                     "ci": ci_half_width(difference), "p": p_value,
                     "n": len(difference)})
    return pd.DataFrame(rows).set_index("window_s")


ride_gap = paired_gap(runs, "kpi.mean_ride_time")

figure, axis = plt.subplots(figsize=(7.4, 4.4))
axis.errorbar(ride_gap.index, ride_gap["mean_gap"], yerr=ride_gap["ci"],
              marker="o", markersize=7, lw=2.0, capsize=4,
              color=SERIES[2], ecolor=INK_SOFT)
axis.axhline(0, lw=1.1, color=INK_SOFT)
axis.set_xlabel("service window (s)")
axis.set_ylabel("L1 − L0 mean ride time (s)")
axis.set_title("Reordering pays more as the window tightens")
axis.set_xticks(WINDOWS)
axis.grid(axis="y")
axis.set_axisbelow(True)
# Annotate the ends, which are the two numbers the section quotes.
for window in (WINDOWS[0], WINDOWS[-1]):
    value = ride_gap.loc[window, "mean_gap"]
    axis.annotate(f"{value:+.0f} s", (window, value),
                  textcoords="offset points", xytext=(6, -14),
                  fontsize=9, color=INK)
axis.text(0.98, 0.06, "below zero = reordering gives shorter rides",
          transform=axis.transAxes, ha="right", fontsize=8.5,
          color=INK_SOFT)
save(figure, "synthetic-window-sweep")
plt.show()

display(Markdown("**L1 − L0, paired by seed.** Negative is better."))
display(ride_gap.assign(p=ride_gap["p"].map("{:.2g}".format)).round(1))
save_table(ride_gap.round(3), "synthetic-window-sweep-table")

assert (ride_gap["mean_gap"] < 0).all(), "L1 no longer wins at every window"
assert (ride_gap["p"] < 1e-4).all(), "a window's advantage lost significance"
print("\nL1 beats L0 on ride at every window, all p < 1e-4 -- "
      f"from {ride_gap['mean_gap'].iloc[-1]:+.0f} s at W1800 "
      f"to {ride_gap['mean_gap'].iloc[0]:+.0f} s at W450")